In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Klasyczne sprzężenie zwrotne i przepływ sterowania (układy dynamiczne)

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



# Klasyczne sprzężenie zwrotne i przepływ sterowania


<details>
<summary><b>Wersje pakietów</b></summary>

Kod na tej stronie został opracowany przy użyciu poniższych wymagań.
Zalecamy korzystanie z tych lub nowszych wersji.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
> **Note:** Nowa wersja układów dynamicznych jest teraz dostępna dla wszystkich użytkowników na wszystkich Backend'ach. Możesz teraz uruchamiać układy dynamiczne w skali narzędziowej. Więcej informacji znajdziesz w [ogłoszeniu](/announcements/product-updates/2025-09-25-new-dynamic-circuits).

Układy dynamiczne to potężne narzędzia, dzięki którym możesz mierzyć Qubity w trakcie wykonywania kwantowego Circuit, a następnie wykonywać klasyczne operacje logiczne w ramach tego Circuit, bazując na wynikach tych pomiarów śródukładowych. Proces ten jest też znany jako _klasyczne sprzężenie zwrotne_. Choć wciąż jesteśmy na wczesnym etapie rozumienia, jak najlepiej wykorzystać układy dynamiczne, społeczność badaczy kwantowych już zidentyfikowała szereg przypadków użycia, takich jak:

* Efektywne przygotowywanie stanów kwantowych, np. [stanu GHZ,](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) [stanu W,](https://arxiv.org/abs/2403.07604) (więcej informacji na temat stanu W znajdziesz w artykule ["State preparation by shallow circuits using feed forward"](https://arxiv.org/abs/2307.14840)) oraz szerokiej klasy [stanów iloczynu macierzowego](https://arxiv.org/abs/2404.16083)
* [Efektywne splątanie dalekiego zasięgu](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) pomiędzy Qubitami na tym samym chipie przy użyciu płytkich Circuit'ów
* Efektywne [próbkowanie układów podobnych do IQP](https://arxiv.org/pdf/2505.04705)

Ulepszenia wprowadzane przez układy dynamiczne wiążą się jednak z pewnymi kompromisami. Pomiary śródukładowe i operacje klasyczne zazwyczaj mają dłuższy czas wykonania niż bramki dwuqubitowe, a ten wzrost czasu może niwelować korzyści wynikające ze zmniejszonej głębokości Circuit'u. Dlatego skracanie czasu pomiarów śródukładowych jest priorytetowym obszarem usprawnień, który IBM Quantum&reg; realizuje w ramach [nowej wersji](/announcements/product-updates/2025-03-03-new-version-dynamic-circuits) układów dynamicznych.

[Specyfikacja OpenQASM 3](https://openqasm.com/language/classical.html#looping-and-branching) definiuje wiele struktur przepływu sterowania, ale Qiskit Runtime obsługuje obecnie tylko warunkową instrukcję `if`. W Qiskit SDK odpowiada jej metoda [if_test](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#if_test) klasy [QuantumCircuit.](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) Metoda ta zwraca [menedżer kontekstu](https://docs.python.org/3/reference/datamodel.html#with-statement-context-managers) i jest zwykle używana w instrukcji `with`. Ten przewodnik opisuje, jak korzystać z tej instrukcji warunkowej.

> **Note:** Przykłady kodu w tym przewodniku używają standardowej instrukcji pomiaru do pomiarów śródukładowych. Zaleca się jednak użycie instrukcji [`MidCircuitMeasure`](/guides/measure-qubits#midcircuit) zamiast niej, jeśli Backend to obsługuje. Szczegółowe informacje znajdziesz w [dokumentacji pomiarów śródukładowych](/guides/measure-qubits#mid-circuit-measurements).
## Instrukcja `if`
Instrukcja `if` służy do warunkowego wykonywania operacji na podstawie wartości klasycznego bitu lub rejestru.

W poniższym przykładzie stosujemy bramkę Hadamarda do Qubitu i mierzymy go. Jeśli wynik wynosi 1, stosujemy bramkę X na tym Qubicie, co powoduje odwrócenie go z powrotem do stanu 0. Następnie mierzymy Qubit ponownie. Wynikowy wynik pomiaru powinien wynosić 0 z prawdopodobieństwem 100%.

In [1]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

circuit.h(q0)
# Use MidCircuitMeasure() if it's supported by the backend.
# circuit.append(MidCircuitMeasure(), [q0], [c0])
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)):
    circuit.x(q0)
circuit.measure(q0, c0)
circuit.draw("mpl")

# example output counts: {'0': 1024}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/60924bfa-50ed-4d9d-a17b-9d64f2cc053f-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/60924bfa-50ed-4d9d-a17b-9d64f2cc053f-0.svg)

Instrukcji `with` można nadać cel przypisania, który sam w sobie jest menedżerem kontekstu — można go zachować i następnie użyć do stworzenia bloku else, który jest wykonywany zawsze wtedy, gdy zawartość bloku `if` *nie* jest wykonywana.

W poniższym przykładzie inicjalizujemy rejestry z dwoma Qubitami i dwoma klasycznymi bitami. Stosujemy bramkę Hadamarda do pierwszego Qubitu i mierzymy go. Jeśli wynik wynosi 1, stosujemy bramkę Hadamarda na drugim Qubicie; w przeciwnym razie stosujemy bramkę X na drugim Qubicie. Na końcu mierzymy też drugi Qubit.

In [2]:
qubits = QuantumRegister(2)
clbits = ClassicalRegister(2)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1) = qubits
(c0, c1) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)) as else_:
    circuit.h(q1)
with else_:
    circuit.x(q1)
circuit.measure(q1, c1)

circuit.draw("mpl")

# example output counts: {'01': 260, '11': 272, '10': 492}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/20f0640a-a3f7-41b3-aada-b66bc89b0555-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/20f0640a-a3f7-41b3-aada-b66bc89b0555-0.svg)

Oprócz warunkowania na pojedynczym klasycznym bicie, możliwe jest również warunkowanie na wartości klasycznego rejestru złożonego z wielu bitów.

W poniższym przykładzie stosujemy bramki Hadamarda do dwóch Qubitów i mierzymy je. Jeśli wynik wynosi `01`, tzn. pierwszy Qubit ma wartość 1, a drugi Qubit ma wartość 0, stosujemy bramkę X do trzeciego Qubitu. Na końcu mierzymy trzeci Qubit. Warto zauważyć, że dla przejrzystości zdecydowaliśmy się podać stan trzeciego klasycznego bitu, który wynosi 0, w warunku `if`. Na rysunku Circuit'u warunek jest wskazywany przez kółka na klasycznych bitach, na których bazuje warunek. Czarne kółko oznacza warunkowanie na 1, natomiast białe kółko oznacza warunkowanie na 0.

In [3]:
qubits = QuantumRegister(3)
clbits = ClassicalRegister(3)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1, q2) = qubits
(c0, c1, c2) = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.if_test((clbits, 0b001)):
    circuit.x(q2)
circuit.measure(q2, c2)

circuit.draw("mpl")

# example output counts: {'101': 269, '011': 260, '000': 252, '010': 243}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/98e8f552-4169-42a3-8182-e14e9ffb59e2-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/98e8f552-4169-42a3-8182-e14e9ffb59e2-0.svg)

## Wyrażenia klasyczne
Moduł klasycznych wyrażeń Qiskit [`qiskit.circuit.classical`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical) zawiera eksperymentalną reprezentację operacji wykonywanych w czasie rzeczywistym na wartościach klasycznych podczas wykonywania Circuit'u. Ze względu na ograniczenia sprzętowe, obsługiwane są obecnie tylko warunki `QuantumCircuit.if_test()`.

Poniższy przykład pokazuje, że możesz użyć obliczania parzystości do tworzenia n-qubitowego stanu GHZ przy użyciu układów dynamicznych. Najpierw wygeneruj $n/2$ par Bella na sąsiednich Qubitach. Następnie połącz te pary warstwą bramek CNOT pomiędzy nimi. Zmierz docelowy Qubit wszystkich poprzednich bramek CNOT i zresetuj każdy zmierzony Qubit do stanu $\vert 0 \rangle$. Zastosuj $X$ do każdego niezmierzonego miejsca, dla którego parzystość wszystkich poprzedzających bitów jest nieparzysta. Na koniec bramki CNOT są stosowane do zmierzonych Qubitów w celu przywrócenia splątania utraconego podczas pomiaru.

W obliczaniu parzystości pierwszy element zbudowanego wyrażenia polega na podniesieniu obiektu Python `mr[0]` do węzła [`Value`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical#value) (`lift` służy do przekształcania dowolnych obiektów w wyrażenia klasyczne). Nie jest to konieczne dla `mr[1]` i ewentualnych kolejnych rejestrów klasycznych, ponieważ są one danymi wejściowymi do `expr.bit_xor`, a wszelkie niezbędne podniesienie jest wykonywane automatycznie w tych przypadkach. Takie wyrażenia można budować w pętlach i innych konstrukcjach.

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.classical import expr

num_qubits = 8
if num_qubits % 2 or num_qubits < 4:
    raise ValueError("num_qubits must be an even integer ≥ 4")
meas_qubits = list(range(2, num_qubits, 2))  # qubits to measure and reset

qr = QuantumRegister(num_qubits, "qr")
mr = ClassicalRegister(len(meas_qubits), "m")
qc = QuantumCircuit(qr, mr)

# Create local Bell pairs
qc.reset(qr)
qc.h(qr[::2])
for ctrl in range(0, num_qubits, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Glue neighboring pairs
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Measure boundary qubits between pairs,reset to 0
for k, q in enumerate(meas_qubits):
    qc.measure(qr[q], mr[k])
    qc.reset(qr[q])

# Parity-conditioned X corrections
# Each non-measured qubit gets flipped iff the parity (XOR) of all
# preceding measurement bits is 1
for tgt in range(num_qubits):
    if tgt in meas_qubits:  # skip measured qubits
        continue
    # all measurement registers whose physical qubit index < tgt
    left_bits = [k for k, q in enumerate(meas_qubits) if q < tgt]
    if not left_bits:  # skip if list empty
        continue

    # build XOR-parity expression
    parity = expr.lift(
        mr[left_bits[0]]
    )  # lift the first bit to Value so it will be treated like a boolean.
    for k in left_bits[1:]:
        parity = expr.bit_xor(
            mr[k], parity
        )  # calculate parity with all other bits
    with qc.if_test(parity):  # Add X if parity is 1
        qc.x(qr[tgt])

# Re-entangle measured qubits
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

In [5]:
qc.draw(output="mpl", style="iqp", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d0f0abdb-50d5-408d-a704-a1a555acdd85-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d0f0abdb-50d5-408d-a704-a1a555acdd85-0.svg)
## Znajdź Backend-i obsługujące dynamiczne Circuit-y
Aby znaleźć wszystkie Backend-i dostępne na Twoim koncie, które obsługują dynamiczne Circuit-y, uruchom kod podobny do poniższego. Ten przykład zakłada, że masz [zapisane dane logowania.](/guides/save-credentials) Możesz też [jawnie podać dane logowania](/guides/initialize-account#explicit) podczas inicjalizacji konta usługi Qiskit Runtime. Pozwoli Ci to na przykład wyświetlić Backend-i dostępne dla konkretnej instancji lub typu planu.

> **Note:** - Backend-i dostępne dla konta zależą od instancji podanej w danych logowania.
> - Nowa wersja dynamicznych Circuit-ów jest teraz dostępna dla wszystkich użytkowników na wszystkich Backend-ach. Więcej szczegółów znajdziesz w [ogłoszeniu](/announcements/product-updates/2025-09-25-new-dynamic-circuits).

In [6]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
dc_backends = service.backends(dynamic_circuits=True)
print(dc_backends)

[<IBMBackend('ibm_pittsburgh')>, <IBMBackend('ibm_boston')>, <IBMBackend('ibm_fez')>, <IBMBackend('ibm_miami')>, <IBMBackend('ibm_marrakesh')>, <IBMBackend('ibm_torino')>, <IBMBackend('ibm_kingston')>]


## Qiskit Runtime limitations

Be aware of the following constraints when running dynamic circuits in Qiskit Runtime.

- Due to the limited physical memory on control electronics, there is also a limit on the number of `if` statements and size of their operands. This limit is a function of the number of broadcasts and the number of broadcasted bits in a job (not a circuit).

   When processing an `if` condition, measurement data needs to be transferred to the control logic to make that evaluation. A broadcast is a transfer of unique classical data, and broadcasted bits is the number of classical bits being transferred. Consider the following:

   ```python
   c0 = ClassicalRegister(3)
   c1 = ClassicalRegister(5)
   ...
   with circuit.if_test((c0, 1)) ...
   with circuit.if_test((c0, 3)) ...
   with circuit.if_test((c1[2], 1)) ...
   ```
   In the previous code example, the first two `if_test` objects on `c0` are considered one broadcast because the content of `c0` did not change, and thus does not need to be re-broadcasted. The `if_test` on `c1` is a second broadcast. The first one broadcasts all three bits in `c0` and the second broadcasts just one bit, making a total of four broadcasted bits.

   Currently, if you broadcast 60 bits each time, then the job can have approximately 300 broadcasts. If you broadcast just one bit each time, however, then the job can have 2400 broadcasts.

- The operand used in an `if_test` statement must be 32 or fewer bits. Thus, if you are comparing an entire `ClassicalRegister`, the size of that `ClassicalRegister` must be 32 or fewer bits. If you are comparing just a single bit from a `ClassicalRegister`, however, that `ClassicalRegister` can be of any size (since the operand is only one bit).

   For example, the "Not valid" code block does not work because `cr` is more than 32 bits.  You can, however, use a classical register wider than 32 bits if you are testing only one bit, as shown in the "Valid" code block.

   <Tabs>
   <TabItem value="Not valid" label="Not valid">
      ```python
         cr = ClassicalRegister(50)
         qr = QuantumRegister(50)
         circuit = QuantumCircuit(qr, cr)
         ...
         circ.measure(qr, cr)
         with circ.if_test((cr, 15)):
            ...
      ```
   </TabItem>
   <TabItem value="Valid" label="Valid">
      ```python
         cr = ClassicalRegister(50)
         qr = QuantumRegister(50)
         circuit = QuantumCircuit(qr, cr)
         ...
         circ.measure(qr, cr)
         with circ.if_test((cr[5], 1)):
            ...
      ```
   </TabItem>
   </Tabs>

- Nested conditionals are not allowed. For example, the following code block will not work because it has an `if_test` inside another `if_test`:
   <Tabs>
    <TabItem value="Not valid" label="Not valid">
        ```python
           c1 = ClassicalRegister(1, "c1")
           c2 = ClassicalRegister(2, "c2")
           ...
           with circ.if_test((c1, 1)):
            with circ.if_test(c2, 1)):
             ...
        ```
     </TabItem>
     <TabItem value="Valid" label="Valid">
        ```python
        cr = ClassicalRegister(2)
        ...
        with circuit.if_test((cr, 0b11)):
          ...
        ```
     </TabItem>
    </Tabs>

- Having `reset` or measurements inside conditionals is not supported.
- Arithmetic operations are not supported.
- See the [OpenQASM 3 feature table](/docs/guides/qasm-feature-table) to determine which OpenQASM 3 features are supported on Qiskit and Qiskit Runtime.
- When OpenQASM 3 (instead of `QuantumCircuit`) is used as the input format to pass circuits to Qiskit Runtime primitives, only instructions that can be loaded into Qiskit are supported. Classical operations, for example, are not supported because they cannot be loaded into Qiskit. See [Import an OpenQASM 3 program into Qiskit](/docs/guides/interoperate-qiskit-qasm3#import-an-openqasm-3-program-into-qiskit) for more information.
- The `for`, `while`, and `switch` instructions are not supported.

## Use dynamic circuits with Estimator

Since Estimator does not support dynamic circuits, you can use Sampler and build your own measurement circuits instead. Alternatively, you can use the [Executor primitive,](/docs/guides/directed-execution-model#executor-primitive) which supports dynamic circuits.

To replicate Estimator's behavior, follow this process:

1. Group the terms of all observables into a partition.  This can be done by using the [`PauliList` API,](/docs/api/qiskit/qiskit.quantum_info.PauliList#group_qubit_wise_commuting) for example.
     <Admonition type="note">
      You can use the [`BitArray`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.primitives.BitArray#expectation_values) primitive attribute to compute the expectation values of the provided observables.
     </Admonition>
2. Execute one basis change circuit per partition (whichever basis change needs to be done for each partition). See the Measurement bases addon utility  [`measurement_bases` module](https://github.com/Qiskit/qiskit-addon-utils/blob/38ea05431f2e9830adf4ec9265f6d19758a32096/qiskit_addon_utils/exp_vals/measurement_bases.py) for more information. [Get started with utilities.](/docs/guides/qiskit-addons-utils#get-started-with-utilities)
3. Add back together the results for each partition.

## Next steps

<Admonition type="tip" title="Recommendations">
- Learn how to implement accurate dynamic decoupling by using [stretch.](/docs/guides/stretch)
- Learn about the shorter [mid-circuit measurements](/docs/guides/measure-qubits#mid-circuit-measurements) that reduce the circuit time.
- Use [circuit schedule visualization](/docs/guides/visualize-circuit-timing#qiskit-runtime-support) to debug and optimize your dynamic circuits.
</Admonition>